In [5]:
import numpy as np
import pandas as pd

# Parameters
S0 = 100.0       # initial price
r = 0.05         # historical arithmetic mean return
s = 0.10         # historical standard deviation (risk)
T = 20            # projection horizon in years
N = 200_000      # number of Monte Carlo paths

# =========================
# Step 1: Convert (r, s) to log‑normal parameters (mu, sigma)
# =========================
sigma2 = np.log((s**2) / (1 + r)**2 + 1)
sigma = np.sqrt(sigma2)
mu = np.log(1 + r) - 0.5 * sigma2

# =========================
# Step 2: Analytical distribution quantiles
# =========================
def analytical_quantile(k: int) -> float:
    """
    k = -2, -1, 0, 1, 2  for   -2σ, -1σ, median, +1σ, +2σ
    returns the corresponding price quantile from the log‑normal model
    """
    return S0 * np.exp(mu + k * sigma)

quantile_labels = ["minus_2σ (≈2.5%)", "minus_1σ (≈16%)",
                   "median (50%)", "plus_1σ (≈84%)", "plus_2σ (≈97.5%)"]
analytical_vals = [analytical_quantile(k) for k in [-2, -1, 0, 1, 2]]

# =========================
# Step 3: Monte Carlo simulation of end‑of‑period price
# =========================
rng = np.random.default_rng(12345)  # for reproducibility
Z = rng.standard_normal(N)
Y = mu + sigma * Z                  # log‑return
S1_sim = S0 * np.exp(Y)             # simulated price at T

mc_quantiles = np.percentile(S1_sim, [2.28, 15.87, 50, 84.13, 97.72])

# =========================
# Step 4: Compare quantiles
# =========================
df_quantiles = pd.DataFrame({
    "Quantile": quantile_labels,
    "Analytical": analytical_vals,
    "Monte Carlo": mc_quantiles,
}).set_index("Quantile")
df_quantiles["Abs. diff"] = (df_quantiles["Analytical"] - df_quantiles["Monte Carlo"]).abs()

# =========================
# Step 5: Recover mean & std of simple returns from simulation
# =========================
R_sim = S1_sim / S0 - 1           # simple return for each path
r_sim = R_sim.mean()
s_sim = R_sim.std(ddof=1)

# =========================
# Output
# =========================
print("Derived log‑normal parameters (1‑period):")
print(f"  mu    = {mu:.8f}")
print(f"  sigma = {sigma:.8f}\n")

print("Quantiles of the price distribution after 1 year:")
print(df_quantiles.to_string(float_format=lambda x: f'{x:,.4f}'))
print()

print("Monte Carlo recovered arithmetic parameters:")
print(f"  mean   r_sim = {r_sim:.6f}   (target r = {r:.6f})")
print(f"  stddev s_sim = {s_sim:.6f}   (target s = {s:.6f})")



Derived log‑normal parameters (1‑period):
  mu    = 0.04427546
  sigma = 0.09502319

Quantiles of the price distribution after 1 year:
                  Analytical  Monte Carlo  Abs. diff
Quantile                                            
minus_2σ (≈2.5%)     86.4356      86.4355     0.0001
minus_1σ (≈16%)      95.0518      95.0954     0.0436
median (50%)        104.5270     104.5553     0.0283
plus_1σ (≈84%)      114.9467     114.9420     0.0048
plus_2σ (≈97.5%)    126.4051     126.4092     0.0041

Monte Carlo recovered arithmetic parameters:
  mean   r_sim = 0.050163   (target r = 0.050000)
  stddev s_sim = 0.099833   (target s = 0.100000)
